# Fortnightly water for a district

Water or not water for every 10 m pixel, once a fortnight, over any date range, from Sentinel-1 and
Sentinel-2 (see `README.md` for the method and its evaluation). Set the area and the dates in the next
cell, then run everything: section 3 is a quick look at weeks of water from 3,000 random pixels, section
4 exports the rasters to Google Drive under `corestack_water/`.

Needs `pip install -r requirements.txt`, `earthengine authenticate` once, and `EE_PROJECT` set to an
Earth Engine enabled Google Cloud project (or `PROJECT` below).

## 1. Area and dates

In [ ]:
DISTRICT = "Jalpaiguri"     # a district in FAO GAUL, or None to use BOUNDARY_FILE
STATE = None                 # only if the district name exists in two states
BOUNDARY_FILE = None         # a GeoJSON / shapefile instead of a district (needs geopandas)
START_DATE = "2023-06-01"
END_DATE = "2024-05-31"      # may be in another year
LAYER = "none"               # none (the model alone), dw, indiasat or both (Section 9 of the README)
PROJECT = None               # Earth Engine project, if EE_PROJECT isn't set

## 2. Build the raster

In [ ]:
import water

ee = water.ee_init(PROJECT)
region = water.district(ee, DISTRICT, STATE) if DISTRICT else water.boundary(ee, BOUNDARY_FILE)
print(f"area: {region.area(1).getInfo() / 1e6:,.0f} km2")

stack = water.water(ee, region, START_DATE, END_DATE, layer=LAYER)
weeks = water.weeks_of_water(stack)
print(f"{len(stack.bandNames().getInfo())} fortnights:", stack.bandNames().getInfo()[:3], "...")

## 3. Quick look

In [ ]:
wet, seen = water.sample_weeks(stack, region, n=3000)
ever = [w for w in wet if w > 0]
print(f"{len(wet)} pixels sampled; {len(ever)} held water at some point ({len(ever) / max(1, len(wet)):.1%})")
if ever:
    ever.sort()
    print(f"weeks of water among them: median {ever[len(ever) // 2]}, most {ever[-1]}")
print(f"weeks observed: median {sorted(seen)[len(seen) // 2]} of {2 * len(stack.bandNames().getInfo())}")

## 4. Export

In [ ]:
name = f"water_{(DISTRICT or 'boundary').lower()}_{START_DATE}_{END_DATE}".replace("-", "")
water.save(ee, stack, region, name, drive=True)
water.save(ee, weeks, region, name + "_weeks", drive=True)
print("follow the two tasks in the Earth Engine Tasks tab: https://code.earthengine.google.com/tasks")